# SARIMAX - SAFEX white maize with external drivers

This notebook replaces the old `ARIMA` model with **`SARIMAX`**, which accepts *exogenous* (external)
variables. The target is the SAFEX white maize price (R/ton); the external drivers are the
**ZAR/USD exchange rate, rainfall and fuel price**.

| Section | What happens |
|---|---|
| 1 | Imports (`ARIMA` -> `SARIMAX`) |
| 2 | Load the SAFEX prices and the external drivers, align them |
| 3 | Chronological train / test split |
| 4 | Fit SARIMAX, forecast the test block, MAE |
| 5 | Do the external factors actually help? (baselines + one-step-ahead) |
| 6 | Tune the `(p, d, q)` order |
| 7 | Forward forecast |

## 1. Update imports and dependencies

In [ ]:
import warnings
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.plotting import autocorrelation_plot

# ARIMA -> SARIMAX (SARIMAX supports exogenous variables)
# from statsmodels.tsa.arima.model import ARIMA        # old import: could only model the price itself
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# pandas (data handling) and scikit-learn (error evaluation)
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
%matplotlib inline

## 2. Load and align your datasets

Expected files (both need a `Date` column):

* `safex_prices.csv` - SAFEX white maize prices, a `Close` column in R/ton (optional Open/High/Low/Volume)
* `sa_factors.csv` - external drivers: `ZAR_USD`, `Rainfall_mm`, `Fuel_Price`

Rename the columns in the settings cell if yours are called something else.

In [ ]:
# ---- Data settings ---------------------------------------------------------
PRICES_FILE = 'safex_prices.csv'   # Date, Close (R/ton)  [+ optional Open, High, Low, Volume]
EXOG_FILE   = 'sa_factors.csv'     # Date, ZAR_USD, Rainfall_mm, Fuel_Price
TARGET_COL  = 'Close'              # SAFEX white maize price column
EXOG_COLS   = ['ZAR_USD', 'Rainfall_mm', 'Fuel_Price']   # external drivers
TRAIN_FRAC  = 0.8                  # chronological split: first 80% train, last 20% test

# ---- SARIMAX settings --------------------------------------------------------
ORDER          = (1, 1, 1)     # (p, d, q) - tuned in section 6
SEASONAL_ORDER = (0, 0, 0, 0)  # (P, D, Q, s) - see the note at the start of section 4
SCALE_EXOG     = False         # set True if you get ConvergenceWarnings (standardises X)

In [ ]:
# Load the SAFEX white maize prices and the South African external drivers.
# (If your dates are written dd/mm/yyyy, add dayfirst=True to both read_csv calls.)
prices = pd.read_csv(PRICES_FILE, parse_dates=['Date'], index_col='Date')
exog_data = pd.read_csv(EXOG_FILE, parse_dates=['Date'], index_col='Date')

# Tidy up: ascending dates and no duplicated dates
prices = prices.sort_index()
prices = prices[~prices.index.duplicated(keep='last')]
exog_data = exog_data.sort_index()
exog_data = exog_data[~exog_data.index.duplicated(keep='last')]

# Fail early, with a readable message, if a column name does not match the settings above
missing = [c for c in [TARGET_COL] if c not in prices.columns] + \
          [c for c in EXOG_COLS if c not in exog_data.columns]
if missing:
    raise KeyError(f'Column(s) not found in the CSV files: {missing}')

In [ ]:
# Safeguard: put lower-frequency drivers (e.g. monthly fuel prices) on a daily calendar BEFORE the
# join, carrying the last known value forward. Otherwise a monthly series only matches ~12 dates a
# year, and the inner join below would silently throw away almost all of the daily price rows.
# (.last().ffill() also fills blanks if all drivers sit in one wide file with empty cells.)
exog_data = exog_data.resample('D').last().ffill()

# Merge both dataframes with an inner join so that the dates align
data = prices.join(exog_data, how='inner')

# Business-day calendar; forward-fill weekend / holiday gaps
data = data.resample('B').ffill()
data = data.dropna(subset=[TARGET_COL] + EXOG_COLS)

In [ ]:
print(f'{len(data)} business days   {data.index.min().date()} -> {data.index.max().date()}')
data[[TARGET_COL] + EXOG_COLS].describe().round(2)

In [ ]:
fig, axes = plt.subplots(len(EXOG_COLS) + 1, 1, figsize=(12, 3 * (len(EXOG_COLS) + 1)), sharex=True)
data[TARGET_COL].plot(ax=axes[0], title=f'SAFEX white maize - {TARGET_COL} (R/ton)')
for ax, col in zip(axes[1:], EXOG_COLS):
    data[col].plot(ax=ax, title=col)
plt.tight_layout()
plt.show()

autocorrelation_plot(data[TARGET_COL])
plt.show()

## 3. Split the data for training and testing

The split is chronological (never shuffled): the model is trained on the earliest 80% of the data and
tested on the most recent 20%.

In [ ]:
# Target variable (y) and external drivers (X)
y = data[TARGET_COL]
X = data[EXOG_COLS]

# Chronological split point (80% train / 20% test)
train_size = int(len(data) * TRAIN_FRAC)

# Training sets
y_train = y.iloc[:train_size]
X_train = X.iloc[:train_size]

# Testing sets
y_test = y.iloc[train_size:]
X_test = X.iloc[train_size:]

print(f'train: {len(y_train)} days  ({y_train.index.min().date()} -> {y_train.index.max().date()})')
print(f'test : {len(y_test)} days  ({y_test.index.min().date()} -> {y_test.index.max().date()})')

In [ ]:
if SCALE_EXOG:
    # Standardise the drivers using TRAIN statistics only (no information from the test block leaks in)
    mu, sigma = X_train.mean(), X_train.std().replace(0, 1)
    X_train = (X_train - mu) / sigma
    X_test = (X_test - mu) / sigma
    X = (X - mu) / sigma          # used by the one-step-ahead evaluation in section 5

### 3b. Diagnostics on the training block

ADF tests whether the price needs differencing (a high p-value on the levels and a low one on the first
difference supports `d = 1`). The ACF / PACF of the differenced series hint at sensible `p` and `q` values.

In [ ]:
for name, series in [('price (levels)', y_train), ('price (1st difference)', y_train.diff().dropna())]:
    stat, pvalue = adfuller(series)[:2]
    print(f'ADF {name:<24s} statistic = {stat:8.3f}   p-value = {pvalue:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y_train.diff().dropna(), lags=40, ax=axes[0], title='ACF of the differenced price')
plot_pacf(y_train.diff().dropna(), lags=40, ax=axes[1], title='PACF of the differenced price')
plt.show()

## 4. Implement and forecast with SARIMAX

`ARIMA` only accepted the price series `y`; `SARIMAX` takes `y` **and** `exog`.

> **Seasonality note.** `SEASONAL_ORDER` is left at `(0, 0, 0, 0)`. With business-day data a weekly season
> is `s = 5` (e.g. `(1, 0, 1, 5)`); an annual season (`s` of about 260) is too heavy for SARIMAX - the Prophet
> notebook models the yearly harvest cycle instead.

> **Read the multi-step MAE with care.** `predict(..., exog=X_test)` is given the *actual* exchange-rate,
> rainfall and fuel values for the whole test block. In real life you would not know them in advance, so
> this score answers *"how well do prices follow the drivers if the drivers are known?"* Judge real
> forecasting skill with the baselines in section 5 and treat future driver values as scenarios (section 7).

In [ ]:
# Initialise the model - pass X_train into the exog parameter
model = SARIMAX(y_train, exog=X_train, order=ORDER, seasonal_order=SEASONAL_ORDER)

# Fit the model
model_fit = model.fit(disp=False)
print(model_fit.summary())

In the summary, the rows named `ZAR_USD`, `Rainfall_mm` and `Fuel_Price` are the exogenous coefficients:
the sign shows the direction of the effect and `P>|z|` whether it is statistically distinguishable from zero.
(If you set `SCALE_EXOG = True` the coefficients are per one standard deviation of each driver.)

In [ ]:
# Residual diagnostics (the first residual is distorted by the differencing initialisation, so skip it)
resid = model_fit.resid.iloc[ORDER[1]:]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
resid.plot(ax=axes[0], title='Residuals')
resid.plot(kind='kde', ax=axes[1], title='Residual density')
plt.show()
print(resid.describe())

model_fit.plot_diagnostics(figsize=(14, 9))
plt.show()

In [ ]:
# Forecast the test block: the testing set's exogenous variables are required for the predictions
predictions = model_fit.predict(start=len(y_train), end=len(y) - 1, exog=X_test)
predictions = pd.Series(np.asarray(predictions), index=y_test.index, name='SARIMAX forecast')
predictions.head()

In [ ]:
def score(y_true, y_pred):
    """MAE (R/ton), RMSE (R/ton) and MAPE (%)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
            'MAPE %': float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)}


results = {}
results['SARIMAX + drivers (multi-step)'] = score(y_test, predictions)

mae = mean_absolute_error(y_test, predictions)
print(f'Mean Absolute Error (MAE): R {mae:,.2f} per ton')

In [ ]:
# Forecast vs actuals with a 95% confidence band
fc = model_fit.get_forecast(steps=len(y_test), exog=X_test)
conf = np.asarray(fc.conf_int(alpha=0.05))
lower = pd.Series(conf[:, 0], index=y_test.index)
upper = pd.Series(conf[:, 1], index=y_test.index)

plt.figure(figsize=(12, 5), dpi=100)
plt.plot(y_train, label='training')
plt.plot(y_test, label='actual')
plt.plot(predictions, label='forecast')
plt.fill_between(y_test.index, lower, upper, color='k', alpha=.15)
plt.title('SARIMAX with external drivers: forecast vs actuals')
plt.ylabel('R / ton')
plt.legend(loc='upper left', fontsize=8)
plt.show()

## 5. Do the external factors actually help?

Three reference points are compared with the model above:

* **SARIMAX without drivers** - same order, but no `exog` (this is the old ARIMA behaviour)
* **Naive** - tomorrow's price equals today's price
* **One-step-ahead** versions - each day is forecast from all prices up to the previous day. The parameters
  are still estimated on the training block only, so there is no peeking at the test data.

In [ ]:
# SARIMAX without the external drivers
base_fit = SARIMAX(y_train, order=ORDER, seasonal_order=SEASONAL_ORDER).fit(disp=False)
base_pred = base_fit.predict(start=len(y_train), end=len(y) - 1)
results['SARIMAX, no drivers (multi-step)'] = score(y_test, base_pred)

# Naive multi-step: repeat the last training price
results['Naive: last training price'] = score(y_test, np.full(len(y_test), y_train.iloc[-1]))

# One-step-ahead: re-use the training-block parameters, filter through the whole sample
one_step = model_fit.apply(endog=y, exog=X).get_prediction(start=train_size).predicted_mean
one_step_base = base_fit.apply(endog=y).get_prediction(start=train_size).predicted_mean
results['SARIMAX + drivers (one-step-ahead)'] = score(y_test, one_step)
results['SARIMAX, no drivers (one-step-ahead)'] = score(y_test, one_step_base)

# Naive one-step: yesterday's price
results['Naive: previous day'] = score(y_test, y.shift(1).iloc[train_size:])

pd.DataFrame(results).T.round(2).sort_values('MAE')

How to read the table:

* If **SARIMAX + drivers** has a lower MAE than **SARIMAX, no drivers**, the external factors add information.
* Neither number is impressive unless it also beats the matching **Naive** line - for commodity prices
  the previous day's price is a hard benchmark, especially one step ahead.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_test, label='actual', color='black', linewidth=1)
plt.plot(predictions, label='multi-step (with drivers)')
plt.plot(pd.Series(np.asarray(one_step), index=y_test.index), label='one-step-ahead (with drivers)')
plt.title('Test block: multi-step vs one-step-ahead SARIMAX')
plt.ylabel('R / ton')
plt.legend()
plt.show()

## 6. Tune the `(p, d, q)` order

A small grid search over `p` and `q` (with `d` kept from `ORDER`), ranked by AIC **on the training block
only**. If a combination fails to converge it is skipped.

In [ ]:
d = ORDER[1]
rows = []
for p, q in itertools.product(range(0, 4), range(0, 4)):
    try:
        res = SARIMAX(y_train, exog=X_train, order=(p, d, q),
                      seasonal_order=SEASONAL_ORDER).fit(disp=False)
        rows.append({'order': (p, d, q), 'AIC': res.aic, 'BIC': res.bic})
    except Exception as err:                       # non-invertible / failed fits
        print(f'order {(p, d, q)} skipped: {err}')

order_table = pd.DataFrame(rows).sort_values('AIC').reset_index(drop=True)
order_table.head(10)

In [ ]:
best_order = order_table.loc[0, 'order']
print('Best order by AIC:', best_order)

tuned_fit = SARIMAX(y_train, exog=X_train, order=best_order,
                    seasonal_order=SEASONAL_ORDER).fit(disp=False)

tuned_pred = tuned_fit.predict(start=len(y_train), end=len(y) - 1, exog=X_test)
tuned_one_step = tuned_fit.apply(endog=y, exog=X).get_prediction(start=train_size).predicted_mean

results[f'SARIMAX {best_order} + drivers (multi-step)'] = score(y_test, tuned_pred)
results[f'SARIMAX {best_order} + drivers (one-step-ahead)'] = score(y_test, tuned_one_step)

pd.DataFrame(results).T.round(2).sort_values('MAE')

## 7. Forward forecast

To forecast days that have not happened yet you need *future values of the drivers*, which you do not have.
The usual approach is to feed in **scenarios**. Below the last observed values are carried forward; edit
`future_exog` to test, for example, a weaker rand or a higher fuel price.

In [ ]:
N_AHEAD = 20     # business days

final_fit = SARIMAX(y, exog=X, order=best_order, seasonal_order=SEASONAL_ORDER).fit(disp=False)

future_index = pd.bdate_range(y.index[-1] + pd.offsets.BDay(1), periods=N_AHEAD)
future_exog = pd.DataFrame({col: X[col].iloc[-1] for col in EXOG_COLS}, index=future_index)
# Example scenario: rand 5% weaker -> future_exog['ZAR_USD'] *= 1.05   (if SCALE_EXOG is False)

future_fc = final_fit.get_forecast(steps=N_AHEAD, exog=future_exog).summary_frame(alpha=0.05)
future_fc.index = future_index
future_fc[['mean', 'mean_ci_lower', 'mean_ci_upper']].round(2)